# N099 · 一维DP与以当前位置结尾的状态

**目标：** 区分前缀最优和必须选当前位置的最优。

**先修：** N098, N015。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 线性递推；House Robber；Kadane；滚动变量；负数基例。

**配套讲解来源：** [同名逐章意见](../../comment/099_linear_dp_and_kadane.md)。

本章阅读顺序：**先明白问题 → 补基础 → 手算 → 看核心片段 → 分段运行 → 核对测试**。需要复制使用时，跳到末尾“完整代码”，它包含全部依赖、实现和测试。

## 1. 通俗解释：输入什么，要得到什么

这一章其实讲了三个紧密相关的问题，它们共享同一种"沿着数组从左到右推"的思路：

1. **打家劫舍（线性）**：一排房子，每间有现金，你不能偷相邻的两间，问最多能偷多少。比如输入 `[2,7,9,3,1]`，输出是 `12`，因为选第 1、3、5 间（值 2+9+1=12）是最优的；选 7+3=10 或 7+1=8 都不如它。
2. **打家劫舍（环形）**：房子排成一个圈，第一间和最后一间也算相邻。比如输入 `[2,3,2]`，输出是 `3`：看起来 2+2=4 更大，但首尾两间在环上相邻，不能同时选，所以只能偷 3 这一间。
3. **最大子数组**：在数组里挑一段**连续**的非空区间，让区间内数字之和最大。比如输入 `[-2,1,-3,4,-1,2,1,-5,4]`，输出是 `6`，因为子数组 `[4,-1,2,1]` 的和是 6，没有任何别的连续段比它更大。

这三个问题的共同点是：答案可以由"前一个位置的最优信息"推出来，所以一趟循环就能解决。本章的核心教学点是要你分清两种状态——"前 i 个元素里的最优"和"必须以第 i 个元素结尾的最优"，它们长得像，但用法完全不同。

## 2. 基础知识：先读懂这些词

- **线性 DP（一维 DP）**：状态只沿一个方向（数组下标）推进的动态规划。你可以把它理解为"从左往右填一张一维表格，每一格只依赖前面几格"。
- **前缀最优状态**：状态表示"只看前 i 个元素时，满足约束的最优答案"。它不关心最后一个元素选没选，所以需要两个变量分别记"选了末尾"和"没选末尾"两种情况。
- **以当前位置结尾的状态**：状态表示"必须选上第 i 个元素时的最优答案"。最大子数组要的区间是连续的，所以必须知道"以 i 结尾的区间最好是多少"，否则接不上下一个元素。
- **滚动变量（rolling variables）**：当新状态只依赖最近一两个旧状态时，我们不需要存整张 DP 表，用两三个变量像接力棒一样往后传就行。空间从 O(n) 降到 O(1)。
- **Kadane 算法**：求最大子数组的经典算法，就是上面"以 i 结尾"状态的具体化：要么从 i 重新开始，要么把前面接过来，两者取大。
- **环形问题拆解**：房子成环时首尾冲突，直接线性推会出错。技巧是：任何合法方案要么不选第一间，要么不选最后一间，所以跑两次线性 rob（一次去掉末屋、一次去掉首屋），取较大者。
- **非空约束**：rob 允许一间都不偷（答案至少为 0），而最大子数组要求区间非空（全负数组要返回最大的那个负数，而不是 0）。这个差别是本章反复强调的"失效条件"。
- **暴力对拍**：用一个肯定正确但很慢的枚举解法（把所有方案列出来）去对照快速解法的结果，是验证 DP 正确性的实用手段，“自动测试”部分 里就写了一个。

## 3. 思路推导：从小例子开始

**先看线性打家劫舍 `rob`：**

- Step 1：把问题想成"逐间房子决定偷或不偷"。约束只有一条：相邻不能都偷。
- Step 2：用两个变量描述走到第 i 间时的状态：`before` 表示"前 i−1 间房的最优收获"（即不偷第 i 间时能拿到的），`prev` 表示"前 i 间房的最优收获"。
- Step 3：转移时只有两种选择——不偷第 i 间，收获就是 `prev`（沿用前一间的结果）；偷第 i 间，那么第 i−1 间一定不能偷，收获是 `before + x`。取两者较大值作为新的 `prev`，而旧的 `prev` 变成新的 `before`。
- Step 4：循环结束时的 `prev` 就是全局最优。

手算一遍 `rob([2,7,9,3,1])`：

| 看到房子 x | before（上一轮 prev） | prev = max(prev, before+x) |
|---|---|---|
| 初始 | 0 | 0 |
| 2 | 0 | max(0, 0+2) = 2 |
| 7 | 2 | max(2, 0+7) = 7 |
| 9 | 7 | max(7, 2+9) = 11 |
| 3 | 11 | max(11, 7+3) = 11 |
| 1 | 11 | max(11, 11+1) = 12 |

最后 `prev = 12`，对应偷第 1、3、5 间（2+9+1）。

**再看环形 `rob_circular`：**

- Step 1：环上唯一的新麻烦是"第一间和最后一间相邻"。
- Step 2：任何一个合法方案都不可能同时包含第一间和最后一间，所以所有方案分成两批：不含最后一间的（在 `nums[:-1]` 上做线性 rob），以及不含第一间的（在 `nums[1:]` 上做线性 rob）。
- Step 3：两批合起来覆盖了全部合法方案，所以答案就是两次线性 rob 的较大值。
- Step 4：特殊情况是只有 0 或 1 间房：此时不存在"首尾冲突"，直接返回那间房的值（空数组返回 0）。

**最后看最大子数组 `max_subarray`（Kadane）：**

- Step 1：区间必须连续，所以定义 `ending` 为"必须以当前位置结尾的最大区间和"。
- Step 2：走到新元素 x 时只有两个选择：要么把 x 单独作为新区间开头（前面的累积是负担时），要么把 x 接在以左邻居结尾的最优区间后面。即 `ending = max(x, ending + x)`。
- Step 3：全局答案 `best` 是所有 `ending` 的最大值，因为最优区间总得以某个位置结尾。

“运行示例”部分 里正好演示了这个过程，数组是 `[-2,1,-3,4,-1,2,1,-5,4]`，表格每一行是（位置、值、以此结尾、全局最优）：

| 位置 | 值 | 以此结尾 ending | 全局最优 best |
|---|---|---|---|
| 0 | -2 | -2 | -2 |
| 1 | 1 | max(1, -2+1)=1 | 1 |
| 2 | -3 | max(-3, 1-3)=-2 | 1 |
| 3 | 4 | max(4, -2+4)=4 | 4 |
| 4 | -1 | max(-1, 4-1)=3 | 4 |
| 5 | 2 | max(2, 3+2)=5 | 5 |
| 6 | 1 | max(1, 5+1)=6 | 6 |
| 7 | -5 | max(-5, 6-5)=1 | 6 |
| 8 | 4 | max(4, 1+4)=5 | 6 |

你可以看到位置 3 那一步：前面的累积是 −2（负数），带上它还不如从 4 重新开始，所以 `ending` 直接重启为 4。最终 `best = 6`，对应的区间是 `[4,-1,2,1]`。

## 4. 核心代码：rob

先关注决定答案的状态与更新操作。这是完整实现的对应片段，不是另一套解法；导入、辅助结构及其余接口在下一节补全。

```python
def rob(nums):
    before = prev = 0
    for x in nums:
        before, prev = (prev, max(prev, before + x))
    return prev
```

### 逐段读懂代码

**函数一：`rob`（线性打家劫舍）**

```python
def rob(nums):
    before = prev = 0
    for x in nums:
        before, prev = (prev, max(prev, before + x))
    return prev
```

- `before=prev=0`：一开始一间房都没看，两种状态的收获都是 0。这也意味着"一间都不偷"是合法的，全 0 或负收益时答案至少是 0。
- `before,prev=prev,max(prev,before+x)`：这是 Python 的元组同步赋值，右边先全部算完再一次性赋值，所以新 `prev` 用的是**旧的** `before`，不会互相污染。新 `before` 接过旧 `prev`（往前滚动一格），新 `prev` 在"不偷这间（旧 prev）"和"偷这间（旧 before + x）"里取大。
- `return prev`：循环跑完，`prev` 就是"全部房子的最优收获"。

**函数二：`rob_circular`（环形打家劫舍）**

```python
def rob_circular(nums):
    if len(nums) <= 1:
        return max([0] + list(nums))
    return max(rob(nums[:-1]), rob(nums[1:]))
```

- `if len(nums)<=1`：挡住 0 间和 1 间房这两种特殊情况。此时没有"首尾相邻"的问题，1 间房就返回它的值，0 间房（空列表）时 `[0]+list(nums)` 只有 `[0]`，`max` 返回 0。
- `max([0]+list(nums))`：把 0 拼进候选里，空输入也有合法返回值 0。
- `rob(nums[:-1])`：切掉最后一间，跑线性版本，代表"一定不偷末屋"的所有方案。
- `rob(nums[1:])`：切掉第一间，代表"一定不偷首屋"的所有方案。
- 外层 `max`：两批方案取较优者。注意切片会复制数组，所以这个实现用了 O(n) 额外空间（“正确性与复杂度”部分 里也提到了，可以改成传下标范围来消除）。

**函数三：`max_subarray`（Kadane 最大子数组）**

```python
def max_subarray(nums):
    if not nums:
        raise ValueError('nonempty array required')
    ending = best = nums[0]
    for x in nums[1:]:
        ending = max(x, ending + x)
        best = max(best, ending)
    return best
```

- `if not nums: raise ValueError(...)`：明确挡掉空数组。最大子数组要求区间非空，空输入没有定义好的答案，直接报错比悄悄返回 0 更诚实（这正是与 rob 的关键区别）。
- `ending=best=nums[0]`：用第一个元素初始化。不能用 0 初始化，否则全负数组（比如 `[-3,-1,-2]`）会错误地返回 0。
- `for x in nums[1:]`：从第二个元素开始扫。
- `ending=max(x,ending+x)`："以 x 结尾"要么单独开新段，要么接上旧段，取大者。
- `best=max(best,ending)`：每走一步都更新全局最优。
- 分号把两句话写在一行，这只是紧凑写法，逻辑上就是先更新 `ending` 再更新 `best`。

## 5. 分段实现：按顺序运行

**本章接口：** `rob(nums)`、`rob_circular(nums)`、`max_subarray(nums)`

### rob

In [1]:
def rob(nums):
    before = prev = 0
    for x in nums:
        before, prev = (prev, max(prev, before + x))
    return prev

### rob_circular

In [2]:
def rob_circular(nums):
    if len(nums) <= 1:
        return max([0] + list(nums))
    return max(rob(nums[:-1]), rob(nums[1:]))

### max_subarray

In [3]:
def max_subarray(nums):
    if not nums:
        raise ValueError('nonempty array required')
    ending = best = nums[0]
    for x in nums[1:]:
        ending = max(x, ending + x)
        best = max(best, ending)
    return best

## 6. 运行示例与状态可视化

先预测结果再运行。手算表帮助解释过程；下面的表由当前实现计算。若表只展示输入输出，它并不代替内部状态轨迹。

In [4]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

a=[-2,1,-3,4,-1,2,1,-5,4]; ending=best=a[0]; rows=[(0,a[0],ending,best)]
for i,x in enumerate(a[1:],1): ending=max(x,ending+x); best=max(best,ending); rows.append((i,x,ending,best))
show_table(['位置','值','以此结尾','全局最优'],rows)

位置,值,以此结尾,全局最优
0,-2,-2,-2
1,1,1,1
2,-3,-2,1
3,4,4,4
4,-1,3,4
5,2,5,5
6,1,6,6
7,-5,1,6
8,4,5,6


## 7. 正确性、适用条件与复杂度

所有合法独立集按是否选择末项划分，取两者最大。环上合法集必不同时包含首尾，两个线性子问题覆盖全部可能。非空连续区间若以i结尾，要么从i开始，要么接在以i−1结尾的最优区间后。

**代价：** O(n)时间、线性rob与最大子数组O(1)辅助；环形实现切片O(n)辅助，可改索引范围消除。rob允许不选任何项；max_subarray必须非空。

### 展开理解

**线性 rob 为什么对**：我们按"最后一间偷不偷"把所有可能的方案分成两类。不偷最后一间的方案，其收获恰好等于"前 n−1 间房的最优解"；偷最后一间的方案，倒数第二间必然不偷，其收获等于"前 n−2 间房的最优解"加上最后一间的钱。每个合法方案都落在这两类里（分类没有遗漏），且没有任何方案能同时属于两类（分类没有重复），所以两类各自的最优值取较大者，就是全局最优。初始时 0 间房的收获是 0，之后逐间往右推，每一步都建立在前两步的正确结果上，一路推到底自然正确。

**环形为什么对**：环上多出的唯一约束是"第一间和最后一间不能同时偷"。于是任何合法方案至少满足"不偷第一间"或"不偷最后一间"之一（或都不偷，这不影响归入前面两类）。"不偷末屋"的方案全被 `rob(nums[:-1])` 覆盖，"不偷首屋"的方案全被 `rob(nums[1:])` 覆盖，两批取大就是全体合法方案的最优。

**Kadane 为什么对**：最优区间一定以某个下标 i 结尾。反过来，只要对每个 i 都算出"以 i 结尾的最大区间和"，再取全体最大值，就一定撞上了那个最优区间。而"以 i 结尾"的区间只有两种形态：长度为 1（就是 x 自己），或者长度大于 1（它去掉最后一个元素后是以 i−1 结尾的区间，当然取其中最好的那个）。所以 `max(x, ending+x)` 这一步没有漏掉任何可能。

**复杂度**：三个函数都是一趟循环、每个元素只做常数次比较和加法，所以时间都是 O(n)。n=10⁵ 时大约就是几十万次基本操作，一瞬间就算完；哪怕 n=10⁷ 也就一两秒量级。空间方面，rob 和 max_subarray 只用两个变量，是 O(1)；rob_circular 因为切片复制数组用了 O(n) 额外空间，如果你想消除它，把 rob 改成接收 `(nums, lo, hi)` 下标范围再调用两次即可，思路完全不变。

## 8. 单元测试：每个用例在检查什么

```python
assert rob([2, 7, 9, 3, 1]) == 12 and rob_circular([2, 3, 2]) == 3
```

- `rob([2,7,9,3,1])==12`：这是经典正常用例，最优方案是 2+9+1=12，验证基本转移逻辑。
- `rob_circular([2,3,2])==3`：这是环形的关键陷阱用例。若忽略首尾相邻会算出 4（2+2），正确答案只能是中间的 3，专门检验"拆成两个线性子问题"有没有真的拆对。

```python
assert max_subarray([-3, -1, -2]) == -1
```

- 这是全负数组的特殊用例。它检验你没有把 `best` 初始化成 0——正确答案是最大的那个负数 −1，错误实现会在这里返回 0，暴露"非空约束"没处理好。

```python
for n in range(1, 7):
    for a in product((-2, 0, 3), repeat=n):
        ...
        assert rob(a) == max(linear) and rob_circular(a) == max(circular)
        assert max_subarray(a) == max((sum(a[l:r]) for l in range(n) for r in range(l + 1, n + 1)))
```

- 这是暴力对拍：长度 1 到 6、每个元素取 −2/0/3 三种值的所有组合数组（共 3+9+…+729 个），逐一用二进制 `mask` 枚举每一种选房方案（`mask&(mask<<1)` 用来剔除选了相邻两间的非法方案），算出暴力最优值，再和三个快速函数的结果对照。最大子数组则枚举所有连续区间 `[l:r)` 求和取最大做对照。这相当于在几百个小输入上做了穷举验证，能抓住绝大多数转移写错的情形。

In [5]:
assert rob([2, 7, 9, 3, 1]) == 12 and rob_circular([2, 3, 2]) == 3

assert max_subarray([-3, -1, -2]) == -1

from itertools import product

for n in range(1, 7):
    for a in product((-2, 0, 3), repeat=n):
        linear = []
        circular = []
        for mask in range(1 << n):
            if mask & mask << 1:
                continue
            value = sum((a[i] for i in range(n) if mask >> i & 1))
            linear.append(value)
            if n == 1 or not (mask & 1 and mask >> n - 1 & 1):
                circular.append(value)
        assert rob(a) == max(linear) and rob_circular(a) == max(circular)
        assert max_subarray(a) == max((sum(a[l:r]) for l in range(n) for r in range(l + 1, n + 1)))

print('N099: 所有本章断言通过')

N099: 所有本章断言通过


## 9. 练习：先尝试，再看提示

**练习 1：** 把环拆成互斥首尾约束。

**练习 2：** 返回一条最优选择方案。

<details>
<summary>展开思路提示（不是完整答案）</summary>

- **练习 1（把环拆成互斥首尾约束）**：建议你先写清楚"输入是一个非负整数数组，输出是不偷相邻两间（首尾视为相邻）时的最大收获"。提示：动手验证"不含首屋"和"不含末屋"这两个子问题确实覆盖所有合法方案——想想一个方案如果既含首屋又含末屋会违反什么。再用 `n<=1` 的小数组（比如单间 `[5]`）做边界示例手算，最后可以拿 “自动测试”部分 里那段 mask 暴力枚举当裁判对拍。
- **练习 2（返回一条最优选择方案）**：提示：目前函数只返回最优值，不返回"选了哪几间"。滚动变量把历史丢掉了，所以你需要把每一步的决策（这间偷还是不偷）记录下来，或者把 DP 表存成数组再从末尾回溯：如果 `before+x` 那一支赢了就说明这间被偷了，往前跳两格；否则跳一格。最大子数组同理，可以记下 `best` 是在哪个位置、从哪个起点达成的。

</details>

## 10. 完整代码：可独立运行

下面这一格包含**全部导入、全部接口和验证用例**，不依赖前面的单元格。可以重启内核后只运行这一格，也可以复制到 `.py` 文件后运行。在 Notebook 中 Run All 时会再验证一次，这是为独立复制使用保留的完整版本。

In [6]:
# 完整实现：本单元格不依赖其他单元格。
def rob(nums):
    before = prev = 0
    for x in nums:
        before, prev = (prev, max(prev, before + x))
    return prev

def rob_circular(nums):
    if len(nums) <= 1:
        return max([0] + list(nums))
    return max(rob(nums[:-1]), rob(nums[1:]))

def max_subarray(nums):
    if not nums:
        raise ValueError('nonempty array required')
    ending = best = nums[0]
    for x in nums[1:]:
        ending = max(x, ending + x)
        best = max(best, ending)
    return best

# 示例与回归测试
assert rob([2, 7, 9, 3, 1]) == 12 and rob_circular([2, 3, 2]) == 3

assert max_subarray([-3, -1, -2]) == -1

from itertools import product

for n in range(1, 7):
    for a in product((-2, 0, 3), repeat=n):
        linear = []
        circular = []
        for mask in range(1 << n):
            if mask & mask << 1:
                continue
            value = sum((a[i] for i in range(n) if mask >> i & 1))
            linear.append(value)
            if n == 1 or not (mask & 1 and mask >> n - 1 & 1):
                circular.append(value)
        assert rob(a) == max(linear) and rob_circular(a) == max(circular)
        assert max_subarray(a) == max((sum(a[l:r]) for l in range(n) for r in range(l + 1, n + 1)))

print('N099: 所有本章断言通过')

N099: 所有本章断言通过


## 11. 代表题与迁移

- **198. House Robber（打家劫舍）**：练的正是本章 `rob` 的两变量滚动转移，是最纯粹的线性 DP 入门题。
- **213. House Robber II（打家劫舍 II）**：房子排成环，练的是"把环形约束拆成两个线性子问题再取大"的技巧，对应 `rob_circular`。
- **53. Maximum Subarray（最大子数组）**：Kadane 算法的原型题，练的是"必须以当前位置结尾"的状态设计与非空约束的处理，对应 `max_subarray`。

**题面入口：** [198](https://leetcode.com/problems/house-robber/)

题目用于知识映射；本章完整代码遵循教学接口，不等于每道题的完整平台提交。

## 12. 掌握检查

合上代码，说明输入输出、核心变量的含义、每次更新为什么成立，以及一个边界情况。再完成一道练习，不能只凭“运行通过”判断掌握。

**继续学习：** [课程索引](../../COURSE_INDEX.md)。